# Autoresearch on Google Colab Pro

Port of [Andrej Karpathy's autoresearch](https://github.com/karpathy/autoresearch) for Google Colab Pro.

**What this does:** An AI agent autonomously modifies a GPT training script, trains for 5 minutes, checks if the result improved, keeps or discards, and repeats. You come back to a log of experiments and a better model.

**Colab adaptations:**
- Flash Attention 3 replaced with PyTorch SDPA (works on T4/A100/L4)
- Model size auto-scaled to fit your GPU's VRAM
- `uv` replaced with `pip`
- Agent loop driven by Anthropic API (instead of Claude Code CLI)

---

## Before you begin

1. **Runtime > Change runtime type > GPU** (T4 is fine, A100 is better)
2. You need an **Anthropic API key** for the autonomous agent loop (Cell 5)
3. Colab Pro recommended for longer runtimes (free tier may disconnect)

---
## Cell 1: Environment Setup & GPU Detection

In [ ]:
#@title Cell 1: Install Dependencies & Detect GPU
#@markdown Run this cell first. It installs all dependencies and detects your GPU.

import subprocess, sys, os

# Install dependencies (replacing uv with pip)
packages = [
    "rustbpe>=0.1.0",
    "tiktoken>=0.11.0",
    "pyarrow>=21.0.0",
    "anthropic>=0.49.0",  # For the agent loop
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

# GPU detection
import torch
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime > Change runtime type > GPU"

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
cap = torch.cuda.get_device_capability()

print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem_gb:.1f} GB")
print(f"Compute Capability: {cap[0]}.{cap[1]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

# Classify GPU tier for auto-configuration
if gpu_mem_gb >= 70:
    GPU_TIER = "H100"   # 80GB — original config works
elif gpu_mem_gb >= 35:
    GPU_TIER = "A100"   # 40GB — moderate scaling
elif gpu_mem_gb >= 20:
    GPU_TIER = "L4"     # 24GB — significant scaling
else:
    GPU_TIER = "T4"     # 16GB — aggressive scaling

print(f"\nDetected tier: {GPU_TIER}")
print("Environment ready!")

---
## Cell 2: Clone Repo & Prepare Data

In [ ]:
#@title Cell 2: Clone Autoresearch & Download Data
#@markdown Clones the repo, downloads training shards, and trains the BPE tokenizer.
#@markdown
#@markdown **num_shards**: More shards = more training data variety. 4-10 is fine for experiments.

num_shards = 4  #@param {type:"integer"}

import os

WORK_DIR = "/content/autoresearch"

# Clone if not already present
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/karpathy/autoresearch.git {WORK_DIR}
    print("Cloned autoresearch repo.")
else:
    print("Repo already cloned.")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

# Run data preparation (download shards + train tokenizer)
print(f"\nDownloading {num_shards} shards + training tokenizer...")
!python prepare.py --num-shards {num_shards}

# Verify
cache_dir = os.path.expanduser("~/.cache/autoresearch")
data_files = os.listdir(os.path.join(cache_dir, "data"))
tok_files = os.listdir(os.path.join(cache_dir, "tokenizer"))
print(f"\nData shards: {len([f for f in data_files if f.endswith('.parquet')])}")
print(f"Tokenizer files: {tok_files}")
print("Data preparation complete!")

---
## Cell 3: Patch train.py for Colab GPU Compatibility

This is the critical adaptation cell. It:
1. **Replaces Flash Attention 3** with PyTorch's built-in `scaled_dot_product_attention` (SDPA) — works on all GPUs
2. **Scales model size** to fit your GPU's VRAM
3. **Adjusts batch sizes** to prevent OOM

In [ ]:
#@title Cell 3: Patch train.py for Colab Compatibility
#@markdown Rewrites train.py to work on T4/A100/L4 GPUs.
#@markdown
#@markdown The key change: Flash Attention 3 (Hopper-only) is replaced with
#@markdown PyTorch's native `scaled_dot_product_attention` (SDPA), which works
#@markdown on all CUDA GPUs and automatically picks the best backend.

import os
os.chdir("/content/autoresearch")

# Read original train.py
with open("train.py", "r") as f:
    original = f.read()

# Back up original
with open("train_original.py", "w") as f:
    f.write(original)
print("Backed up original train.py -> train_original.py")

# --- Build the patched train.py ---

# GPU-tier-specific hyperparameters
GPU_CONFIGS = {
    "H100": dict(DEPTH=8,  DEVICE_BATCH_SIZE=128, TOTAL_BATCH_SIZE=2**19, WINDOW_PATTERN='"SSSL"'),
    "A100": dict(DEPTH=8,  DEVICE_BATCH_SIZE=32,  TOTAL_BATCH_SIZE=2**17, WINDOW_PATTERN='"SSSL"'),
    "L4":   dict(DEPTH=6,  DEVICE_BATCH_SIZE=16,  TOTAL_BATCH_SIZE=2**16, WINDOW_PATTERN='"SL"'),
    "T4":   dict(DEPTH=4,  DEVICE_BATCH_SIZE=8,   TOTAL_BATCH_SIZE=2**15, WINDOW_PATTERN='"L"'),
}
cfg = GPU_CONFIGS[GPU_TIER]

patched = original

# 1. Replace the Flash Attention 3 import block with SDPA-based attention
fa3_import_block = """from kernels import get_kernel
cap = torch.cuda.get_device_capability()
# varunneal's FA3 is Hopper only, use kernels-community on non-Hopper GPUs
repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"
fa3 = get_kernel(repo).flash_attn_interface"""

sdpa_import = """# Colab patch: use PyTorch native SDPA instead of Flash Attention 3
# SDPA auto-selects the best backend (FlashAttention-2, Memory-Efficient, or Math)"""

patched = patched.replace(fa3_import_block, sdpa_import)

# 2. Replace the fa3 call in CausalSelfAttention.forward with SDPA
#    Original: y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)
#    SDPA needs (B, heads, T, head_dim) layout and doesn't support sliding window natively,
#    so we use causal=True and skip the window_size (minor quality difference, not a blocker)
old_attn = """        y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)
        y = y.contiguous().view(B, T, -1)"""

new_attn = """        # SDPA expects (B, n_heads, T, head_dim)
        q_sdpa = q.transpose(1, 2)
        k_sdpa = k.transpose(1, 2)
        v_sdpa = v.transpose(1, 2)
        # Expand k,v if using GQA (n_kv_head < n_head)
        if self.n_kv_head < self.n_head:
            rep = self.n_head // self.n_kv_head
            k_sdpa = k_sdpa.repeat_interleave(rep, dim=1)
            v_sdpa = v_sdpa.repeat_interleave(rep, dim=1)
        y = F.scaled_dot_product_attention(q_sdpa, k_sdpa, v_sdpa, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, -1)"""

patched = patched.replace(old_attn, new_attn)

# 3. Scale hyperparameters for GPU tier
import re
patched = re.sub(r'^DEPTH = \d+', f'DEPTH = {cfg["DEPTH"]}', patched, flags=re.MULTILINE)
patched = re.sub(r'^DEVICE_BATCH_SIZE = \d+', f'DEVICE_BATCH_SIZE = {cfg["DEVICE_BATCH_SIZE"]}', patched, flags=re.MULTILINE)
patched = re.sub(r'^TOTAL_BATCH_SIZE = .*$', f'TOTAL_BATCH_SIZE = {cfg["TOTAL_BATCH_SIZE"]}', patched, flags=re.MULTILINE)
patched = re.sub(r'^WINDOW_PATTERN = .*$', f'WINDOW_PATTERN = {cfg["WINDOW_PATTERN"]}', patched, flags=re.MULTILINE)

# 4. Fix H100 peak FLOPS constant to match actual GPU
flops_map = {
    "H100": "989.5e12",
    "A100": "312e12",
    "L4":   "121e12",
    "T4":   "65e12",
}
patched = patched.replace("H100_BF16_PEAK_FLOPS = 989.5e12",
                          f"GPU_PEAK_FLOPS = {flops_map[GPU_TIER]}  # {GPU_TIER}")
patched = patched.replace("H100_BF16_PEAK_FLOPS", "GPU_PEAK_FLOPS")

# 5. For T4: bf16 is not natively supported, use fp16 autocast instead
if GPU_TIER == "T4":
    patched = patched.replace('dtype=torch.bfloat16', 'dtype=torch.float16')
    patched = patched.replace('.bfloat16()', '.half()')

# Write patched version
with open("train.py", "w") as f:
    f.write(patched)

print(f"Patched train.py for {GPU_TIER}:")
print(f"  DEPTH = {cfg['DEPTH']}")
print(f"  DEVICE_BATCH_SIZE = {cfg['DEVICE_BATCH_SIZE']}")
print(f"  TOTAL_BATCH_SIZE = {cfg['TOTAL_BATCH_SIZE']}")
print(f"  WINDOW_PATTERN = {cfg['WINDOW_PATTERN']}")
print(f"  Attention: PyTorch SDPA (replaces FA3)")
if GPU_TIER == 'T4':
    print(f"  Precision: fp16 (T4 lacks native bf16)")
else:
    print(f"  Precision: bf16")
print("\nPatch complete! Ready for baseline run.")

---
## Cell 4: Run Baseline Experiment

This trains the unmodified (but GPU-adapted) model for 5 minutes to establish the starting `val_bpb`. This is the number the agent will try to beat.

In [ ]:
#@title Cell 4: Run Baseline Training (\~5-7 minutes)
#@markdown Runs the patched train.py to establish your baseline val_bpb.
#@markdown The first run is slower due to torch.compile warmup.

import subprocess, os
os.chdir("/content/autoresearch")

print("Running baseline experiment (~5-7 min including compilation)...")
print("="*60)

result = subprocess.run(
    ["python", "train.py"],
    capture_output=True, text=True, timeout=900  # 15 min safety timeout
)

# Save log
with open("run.log", "w") as f:
    f.write(result.stdout)
    if result.stderr:
        f.write("\n--- STDERR ---\n")
        f.write(result.stderr)

if result.returncode != 0:
    print("BASELINE FAILED!")
    print("Last 50 lines of output:")
    lines = (result.stdout + "\n" + result.stderr).strip().split("\n")
    for line in lines[-50:]:
        print(line)
else:
    # Extract metrics
    lines = result.stdout.strip().split("\n")
    print("\n".join(lines[-12:]))  # Print the final summary

    # Parse val_bpb
    val_bpb = None
    peak_vram = None
    for line in lines:
        if line.startswith("val_bpb:"):
            val_bpb = float(line.split()[-1])
        if line.startswith("peak_vram_mb:"):
            peak_vram = float(line.split()[-1])

    if val_bpb:
        print(f"\n{'='*60}")
        print(f"BASELINE val_bpb: {val_bpb:.6f}")
        print(f"Peak VRAM: {peak_vram:.0f} MB ({peak_vram/1024:.1f} GB)")
        print(f"{'='*60}")
        print("\nBaseline established! Ready for autonomous experiments.")

        # Initialize results.tsv
        with open("results.tsv", "w") as f:
            f.write("commit\tval_bpb\tmemory_gb\tstatus\tdescription\n")
            mem_gb = round(peak_vram / 1024, 1)
            f.write(f"baseline\t{val_bpb:.6f}\t{mem_gb}\tkeep\tbaseline\n")
        print("Initialized results.tsv with baseline.")
    else:
        print("Could not parse val_bpb from output. Check run.log.")

---
## Cell 5: Autonomous Agent Loop (Anthropic API)

This is the core autoresearch loop, powered by Claude via the Anthropic API. The agent:
1. Reads the current `train.py` and past results
2. Proposes a modification (architecture, hyperparams, optimizer, etc.)
3. Applies the change
4. Trains for 5 minutes
5. Keeps the change if `val_bpb` improved, discards otherwise
6. Repeats

**You need an Anthropic API key.** Set it below or in Colab Secrets.

In [ ]:
#@title Cell 5: Configure API Key
#@markdown Enter your Anthropic API key. You can also store it in Colab Secrets
#@markdown (key icon in left sidebar) as `ANTHROPIC_API_KEY`.

ANTHROPIC_API_KEY = ""  #@param {type:"string"}

import os

# Try Colab Secrets first, then the manual input
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
    if api_key:
        ANTHROPIC_API_KEY = api_key
        print("Loaded API key from Colab Secrets.")
except:
    pass

if not ANTHROPIC_API_KEY:
    raise ValueError("Set ANTHROPIC_API_KEY above or in Colab Secrets!")

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
print(f"API key set (ends with ...{ANTHROPIC_API_KEY[-4:]})")

In [ ]:
#@title Cell 5b: Run Autonomous Experiment Loop
#@markdown The agent will run experiments autonomously. Each experiment takes ~5-7 minutes.
#@markdown
#@markdown **max_experiments**: How many experiments to run (12 ~= 1 hour)
#@markdown
#@markdown **model**: Which Claude model to use for the agent

max_experiments = 12  #@param {type:"integer"}
model = "claude-sonnet-4-6"  #@param ["claude-sonnet-4-6", "claude-haiku-4-5-20251001", "claude-opus-4-6"]

import anthropic
import subprocess
import json
import re
import os
import time
from datetime import datetime

os.chdir("/content/autoresearch")
client = anthropic.Anthropic()

def read_file(path):
    with open(path, "r") as f:
        return f.read()

def write_file(path, content):
    with open(path, "w") as f:
        f.write(content)

def run_training(timeout=600):
    """Run train.py and return (val_bpb, peak_vram_mb, success)."""
    try:
        result = subprocess.run(
            ["python", "train.py"],
            capture_output=True, text=True, timeout=timeout
        )
        log = result.stdout + "\n" + result.stderr
        write_file("run.log", log)

        if result.returncode != 0:
            return None, None, False, log[-2000:]  # last 2000 chars for error context

        val_bpb = None
        peak_vram = None
        for line in log.split("\n"):
            if line.startswith("val_bpb:"):
                val_bpb = float(line.split()[-1])
            if line.startswith("peak_vram_mb:"):
                peak_vram = float(line.split()[-1])

        if val_bpb is None:
            return None, None, False, log[-2000:]
        return val_bpb, peak_vram, True, ""

    except subprocess.TimeoutExpired:
        return None, None, False, "TIMEOUT: Training exceeded 10 minutes"
    except Exception as e:
        return None, None, False, str(e)

def get_best_bpb():
    """Read results.tsv and return the best (lowest) kept val_bpb."""
    import pandas as pd
    df = pd.read_csv("results.tsv", sep="\t")
    kept = df[df["status"] == "keep"]
    return kept["val_bpb"].min()

def ask_agent_for_modification(train_py, results_tsv, prepare_py, error_context=None):
    """Ask Claude to propose a train.py modification."""
    system_prompt = """You are an autonomous ML researcher modifying a GPT training script to achieve the lowest possible val_bpb (validation bits per byte).

RULES:
- You can ONLY modify train.py. Never change prepare.py.
- The training always runs for a fixed 5-minute time budget.
- Your goal: lower val_bpb. That's the only metric that matters.
- You can change: model architecture, hyperparameters, optimizer settings, batch sizes, model depth, learning rates, activation functions, normalization, attention patterns, etc.
- You CANNOT: add new imports of packages not already available, change the evaluation function.
- Keep changes focused. One idea per experiment.
- IMPORTANT: This runs on a Colab GPU with limited VRAM. Don't make the model too large.
- The attention uses PyTorch SDPA (scaled_dot_product_attention), NOT Flash Attention 3.

Respond with EXACTLY this JSON format (no other text):
{
  "description": "short description of what you're changing and why",
  "train_py": "the complete modified train.py file content"
}"""

    user_msg = f"""Here is the current state:

## prepare.py (READ-ONLY, for reference)
```python
{prepare_py}
```

## Current train.py (you will modify this)
```python
{train_py}
```

## Experiment results so far
```
{results_tsv}
```
"""
    if error_context:
        user_msg += f"""\n## Previous attempt FAILED with this error:\n```\n{error_context}\n```\nPlease fix the issue or try a different approach.\n"""

    user_msg += "\nPropose your next experiment. Return the full modified train.py as JSON."

    response = client.messages.create(
        model=model,
        max_tokens=16000,
        system=system_prompt,
        messages=[{"role": "user", "content": user_msg}]
    )

    text = response.content[0].text

    # Parse JSON from response (handle markdown code blocks)
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]

    parsed = json.loads(text.strip())
    return parsed["description"], parsed["train_py"]


# ─── Main Experiment Loop ───────────────────────────────────────────

prepare_py = read_file("prepare.py")
print(f"Starting autonomous experiment loop ({max_experiments} experiments)")
print(f"Agent model: {model}")
print(f"Current best val_bpb: {get_best_bpb():.6f}")
print("=" * 60)

error_context = None

for exp_num in range(1, max_experiments + 1):
    print(f"\n{'─'*60}")
    print(f"Experiment {exp_num}/{max_experiments}  [{datetime.now().strftime('%H:%M:%S')}]")
    print(f"{'─'*60}")

    best_bpb = get_best_bpb()
    current_train = read_file("train.py")
    results = read_file("results.tsv")

    # 1. Ask agent for a modification
    print("Asking agent for next experiment...")
    try:
        description, new_train = ask_agent_for_modification(
            current_train, results, prepare_py, error_context
        )
        error_context = None  # Clear after successful proposal
    except Exception as e:
        print(f"Agent error: {e}")
        print("Retrying next iteration...")
        error_context = f"Agent API call failed: {e}"
        continue

    print(f"Proposal: {description}")

    # 2. Save backup and apply change
    write_file("train_backup.py", current_train)
    write_file("train.py", new_train)

    # 3. Run training
    print("Training (~5-7 min)...", end="", flush=True)
    t0 = time.time()
    val_bpb, peak_vram, success, err = run_training()
    elapsed = time.time() - t0
    print(f" done ({elapsed:.0f}s)")

    # 4. Evaluate result
    if not success:
        status = "crash"
        val_bpb_str = "0.000000"
        mem_str = "0.0"
        print(f"  CRASH: {err[:200]}")
        # Revert
        write_file("train.py", current_train)
        error_context = err
    elif val_bpb < best_bpb:
        status = "keep"
        val_bpb_str = f"{val_bpb:.6f}"
        mem_str = f"{peak_vram/1024:.1f}"
        improvement = best_bpb - val_bpb
        print(f"  KEEP! val_bpb={val_bpb:.6f} (improved by {improvement:.6f})")
        print(f"  VRAM: {peak_vram:.0f} MB")
    else:
        status = "discard"
        val_bpb_str = f"{val_bpb:.6f}"
        mem_str = f"{peak_vram/1024:.1f}"
        print(f"  DISCARD: val_bpb={val_bpb:.6f} (best={best_bpb:.6f})")
        # Revert
        write_file("train.py", current_train)

    # 5. Log result
    with open("results.tsv", "a") as f:
        desc_safe = description.replace("\t", " ").replace("\n", " ")[:100]
        f.write(f"exp{exp_num:03d}\t{val_bpb_str}\t{mem_str}\t{status}\t{desc_safe}\n")

    print(f"  Logged to results.tsv")

print(f"\n{'='*60}")
print(f"Experiment loop complete! {max_experiments} experiments run.")
print(f"Final best val_bpb: {get_best_bpb():.6f}")
print(f"Results saved to results.tsv")
print(f"{'='*60}")

---
## Cell 6: Analyze Results

In [ ]:
#@title Cell 6: Visualize Experiment Results
#@markdown Plots val_bpb over time, shows kept vs discarded experiments, and prints summary.

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

os.chdir("/content/autoresearch")

df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.lower()

print(f"Total experiments: {len(df)}")
print(f"Kept: {(df['status']=='keep').sum()}  |  Discarded: {(df['status']=='discard').sum()}  |  Crashed: {(df['status']=='crash').sum()}")
print()

# --- Summary ---
kept = df[df["status"] == "keep"]
if len(kept) > 0:
    baseline = kept.iloc[0]["val_bpb"]
    best = kept["val_bpb"].min()
    best_row = kept.loc[kept["val_bpb"].idxmin()]
    improvement = baseline - best
    print(f"Baseline val_bpb:  {baseline:.6f}")
    print(f"Best val_bpb:      {best:.6f}")
    print(f"Total improvement: {improvement:.6f} ({improvement/baseline*100:.2f}%)")
    print(f"Best experiment:   {best_row['description']}")
    print()

    # --- Kept experiments list ---
    print("Kept experiments (improvements that stuck):")
    for i, row in kept.iterrows():
        print(f"  #{i:3d}  bpb={row['val_bpb']:.6f}  mem={row['memory_gb']:.1f}GB  {row['description']}")
    print()

# --- Plot ---
fig, ax = plt.subplots(figsize=(14, 7))

valid = df[df["status"] != "crash"].reset_index(drop=True)

if len(valid) > 0:
    # Discarded points
    disc = valid[valid["status"] == "discard"]
    ax.scatter(disc.index, disc["val_bpb"], c="#cccccc", s=30, alpha=0.6,
               zorder=2, label="Discarded")

    # Kept points
    kept_v = valid[valid["status"] == "keep"]
    ax.scatter(kept_v.index, kept_v["val_bpb"], c="#2ecc71", s=80, zorder=4,
               label="Kept", edgecolors="black", linewidths=0.5)

    # Running minimum
    if len(kept_v) > 0:
        running_min = kept_v["val_bpb"].cummin()
        ax.step(kept_v.index, running_min, where="post", color="#27ae60",
                linewidth=2.5, alpha=0.7, zorder=3, label="Running best")

    # Labels on kept experiments
    for idx in kept_v.index:
        row = valid.loc[idx]
        desc = str(row["description"])[:40]
        ax.annotate(desc, (idx, row["val_bpb"]),
                    textcoords="offset points", xytext=(6, 8),
                    fontsize=7.5, color="#1a7a3a", alpha=0.9,
                    rotation=25, ha="left", va="bottom")

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch on Colab ({GPU_TIER}) — {len(df)} Experiments", fontsize=14)
ax.legend(loc="upper right", fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved progress.png")

---
## Cell 7: Download Results (Optional)

Download your experiment results and the best `train.py` to your local machine.

In [ ]:
#@title Cell 7: Download Results
#@markdown Downloads results.tsv, progress.png, and the best train.py to your machine.

from google.colab import files
import os
os.chdir("/content/autoresearch")

for f in ["results.tsv", "progress.png", "train.py"]:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded {f}")